# 02. Dataset Understanding — Schema, Dtypes & Memory Profiling

Before diving into feature engineering, an engineer must achieve complete visibility into a dataset's physical and logical schema:
- **Dimensionality**: Observation count $N$ vs Feature count $P$.
- **Attribute Scale Taxonomy**: Nominal, Ordinal, Interval, and Ratio.
- **Memory Footprint**: Cache pressure, object references, and numeric downcasting opportunities.

In [1]:
import numpy as np
import pandas as pd

# Synthesize a multi-type enterprise transactions dataset
n = 50000
rng = np.random.default_rng(seed=101)

df_trans = pd.DataFrame({
    'trans_id': np.arange(100001, 100001 + n, dtype=np.int64),
    'customer_id': rng.integers(1000, 5000, size=n, dtype=np.int64),
    'branch': rng.choice(['North', 'South', 'East', 'West', 'Central'], size=n),
    'item_category': rng.choice(['Electronics', 'Groceries', 'Clothing', 'Home', 'Beauty'], size=n),
    'quantity': rng.integers(1, 15, size=n, dtype=np.int64),
    'unit_price': rng.uniform(5.0, 500.0, size=n).round(2),
    'tax_rate': rng.choice([0.05, 0.08, 0.12], size=n),
    'is_loyalty': rng.choice([True, False], size=n)
})

print(f"Dataset shape: {df_trans.shape} (Rows: {df_trans.shape[0]:,}, Columns: {df_trans.shape[1]})")

Dataset shape: (50000, 8) (Rows: 50,000, Columns: 8)

## 1. Deep Memory Footprint Analysis

By default, Pandas assigns 64-bit integer (`int64`) and 64-bit float (`float64`) types, while strings default to pointer-based `object` arrays, consuming massive amounts of RAM.

In [1]:
# Inspect baseline memory footprint with deep inspection
initial_memory_bytes = df_trans.memory_usage(deep=True).sum()
initial_memory_mb = initial_memory_bytes / (1024 ** 2)

print("Baseline Column Memory Breakdown:")
mem_series = df_trans.memory_usage(deep=True) / (1024 ** 2)
for col, mem in mem_series.items():
    print(f"  - {col:<15}: {mem:,.2f} MB")
print(f"\nTotal Initial Memory Usage: {initial_memory_mb:,.2f} MB")

Baseline Column Memory Breakdown:
  - Index          : 0.00 MB
  - trans_id       : 0.38 MB
  - customer_id    : 0.38 MB
  - branch         : 3.12 MB
  - item_category  : 3.24 MB
  - quantity       : 0.38 MB
  - unit_price     : 0.38 MB
  - tax_rate       : 0.38 MB
  - is_loyalty     : 0.05 MB

Total Initial Memory Usage: 8.32 MB

## 2. Type Downcasting & Memory Optimization

We can downcast:
- `int64` to `int32` or `int16`
- High-cardinality strings to `category`
- `float64` to `float32`

In [1]:
# Apply systematic downcasting
df_opt = df_trans.copy()

# Downcast integers
df_opt['trans_id'] = df_opt['trans_id'].astype(np.int32)
df_opt['customer_id'] = df_opt['customer_id'].astype(np.int16)
df_opt['quantity'] = df_opt['quantity'].astype(np.int8)

# Downcast floats
df_opt['unit_price'] = df_opt['unit_price'].astype(np.float32)
df_opt['tax_rate'] = df_opt['tax_rate'].astype(np.float32)

# Convert low-cardinality strings to categorical
df_opt['branch'] = df_opt['branch'].astype('category')
df_opt['item_category'] = df_opt['item_category'].astype('category')

opt_memory_mb = df_opt.memory_usage(deep=True).sum() / (1024 ** 2)
savings_pct = (1.0 - opt_memory_mb / initial_memory_mb) * 100

print(f"Optimized Memory Usage: {opt_memory_mb:,.2f} MB")
print(f"Memory Footprint Reduction: {savings_pct:,.1f}% reduction achieved!")

Optimized Memory Usage: 1.05 MB
Memory Footprint Reduction: 87.4% reduction achieved!

## 3. Attribute Scale Classification
Data attributes fall into four distinct measurement levels:
1. **Nominal**: Labels without intrinsic ordering (e.g., `branch`, `item_category`).
2. **Ordinal**: Categorical data with logical progression (e.g., Education: High School < Bachelor < Master).
3. **Interval**: Numeric data where differences are meaningful but no absolute zero exists (e.g., Temperature in Celsius).
4. **Ratio**: Numeric data with a true physical zero where division is meaningful (e.g., `quantity`, `unit_price`).